## Task 4.1 — One validated call

In [1]:
!pip install -q google-genai pydantic

In [2]:
import json
import time
import pandas as pd
from google import genai
from google.colab import userdata
from google.genai import types
from pydantic import BaseModel, Field, ValidationError


In [3]:
ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

SCHEMA = {
    "type" : "object",
    "properties" : {
        "root_cause" : {"type" : "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"]

}

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

MODEL_NAME = "gemini-3.6-flash"

SYSTEM_TEXT = """
You are a clinical operations analyst for CareLoop Clinics.   You extract structured insights from call-centre follow-up notes in valid JSON.
"""

api_call_count = 0

def build_craft_prompt(note_text):
  return f"""
Context: CareLoop Clinics runs outpatient clinics across Indian cities. After a patient misses an appointment, the call-centre phones them and writes a short note.
Role: Act as a careful healthcare operations analyst categorizing missed-appointment reasons.
Action: Read the follow-up note below and extract:
  1. root_cause: Pick EXACTLY ONE value from this allowed list: {', '.join(ALLOWED_CAUSES)}.
     - If the note is ambiguous, unclear, or does not clearly match a specific reason, output "other" rather than guessing.
     - Understand Hinglish phrases naturally (e.g., "bhool gaya" or "yaad nahi raha" means forgot_or_no_reminder).
  2. confidence: A number between 0 and 1 reflecting how clearly the note states the reason.
  3. summary: A concise summary in at most 25 words.
  4. recommended_action: One practical next step for the clinic front desk.
Format: Return a JSON object with exactly four keys: root_cause, confidence, summary, recommended_action.
Tone: Objective, concise, and operational.

Call-centre note: "{note_text}"
"""

def analyse_note(note_text, return_raw=False):
  global api_call_count
  prompt = build_craft_prompt(note_text)
  api_call_count += 1
  response = client.models.generate_content(
      model = MODEL_NAME,
      contents = prompt,
      config=types.GenerateContentConfig(
          system_instruction = SYSTEM_TEXT,
          temperature = 0,
          max_output_tokens = 1000,
          response_mime_type = "application/json",
          response_schema = SCHEMA,
          thinking_config = types.ThinkingConfig(thinking_level="minimal")
      )
  )
  raw_text  = response.text

  if return_raw:
    return raw_text
  return json.loads(raw_text)

notes_df = pd.read_csv("data/followup_notes.csv")
n001_text = notes_df.loc[notes_df["note_id"] == "N001", "note_text"].iloc[0]

n001_result = analyse_note(n001_text)
print("Note N001 text:", n001_text)
print("Parsed dictionary:", n001_result)

Note N001 text: Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.
Parsed dictionary: {'root_cause': 'forgot_or_no_reminder', 'confidence': 0.98, 'summary': 'Patient forgot the appointment due to receiving no reminder message and requested to rebook next week.', 'recommended_action': "Reschedule the appointment for next week and verify the patient's phone number for SMS reminders."}


## Task 4.2 — Validate and process the 10 notes

In [4]:
#Pydantic Model
class FollowupInsight(BaseModel):
  root_cause : str = Field(min_length=3)
  confidence : float = Field(ge=0, le=1)
  summary : str = Field(min_length=5)
  recommended_action : str = Field(min_length=5)

first_10 = notes_df.head(10)
records = []

for idx, row in first_10.iterrows():
  note_id = row["note_id"]
  appt_id = row["appointment_id"]
  note_text = row["note_text"]

  if pd.isna(note_text) or str(note_text).strip() == "":
    records.append({
        "note_id": note_id,
        "appointment_id": appt_id,
        "root_cause": "other",
        "confidence": 0.0,
        "summary": "NO note recorded",
        "recommended_action": "Re-call patient",
        "validation_status": "skipped_blank",
        "raw_response": ""
    })
    print(f"{note_id}: skipped blank (no API call)")
    continue

  raw_text = ""
  parsed_data = {}
  status = "invalid"

  try:
    raw_text = analyse_note(note_text, return_raw=True)
    parsed_data = json.loads(raw_text)

    validated = FollowupInsight(**parsed_data)
    if validated.root_cause in ALLOWED_CAUSES:
      status = "valid"
    else:
      status = "invalid"

  except (ValidationError, json.JSONDecodeError, Exception) as e:
    print(f"{note_id} failed validation or call {e}")
    status = "invalid"

  records.append({
      "note_id": note_id,
      "appointment_id": appt_id,
      "root_cause": parsed_data.get("root_cause", "") if isinstance(parsed_data, dict) else "",
      "confidence": parsed_data.get("confidence", "") if isinstance(parsed_data, dict) else "",
      "summary": parsed_data.get("summary", "") if isinstance(parsed_data, dict) else "",
      "recommended_action": parsed_data.get("recommended_action", "") if isinstance(parsed_data, dict) else "",
      "validation_status": status,
      "raw_response": raw_text
  })
  print(f"{note_id}: {status} -> {records[-1]['root_cause']}")

  time.sleep(13)

# save notes_analysed.csv
analysed_df = pd.DataFrame(records, columns=["note_id", "appointment_id", "root_cause", "confidence", "summary", "recommended_action", "validation_status", "raw_response"])
analysed_df.to_csv("notes_analysed.csv", index=False)

print(f"\nSaved notes_analysed.csv with shape: {analysed_df.shape}")
display(analysed_df[["note_id", "appointment_id", "root_cause", "confidence", "validation_status"]])

N001: valid -> forgot_or_no_reminder
N002: valid -> distance_or_transport
N003: valid -> cost_or_fee
N004: skipped blank (no API call)
N005: valid -> illness_or_emergency
N006: valid -> work_or_schedule
N007: valid -> forgot_or_no_reminder
N008: valid -> distance_or_transport
N009: valid -> other
N010: valid -> work_or_schedule

Saved notes_analysed.csv with shape: (10, 8)


,note_id,appointment_id,root_cause,confidence,validation_status
0,N001,A0872,forgot_or_no_reminder,0.98,valid
1,N002,A0897,distance_or_transport,0.95,valid
2,N003,A0526,cost_or_fee,0.98,valid
3,N004,A0926,other,0.00,skipped_blank
4,N005,A0534,illness_or_emergency,0.95,valid
5,N006,A0507,work_or_schedule,0.95,valid
6,N007,A0333,forgot_or_no_reminder,0.95,valid
7,N008,A0222,distance_or_transport,0.95,valid
8,N009,A0977,other,0.90,valid
9,N010,A0596,work_or_schedule,0.95,valid


## Task 4.3 — Break it on purpose

In [5]:
print(" (a) Pydantic Validation Tests")

bad_confidence = {
    "root_cause": "cost_or_fee",
    "confidence": 1.7,
    "summary": "Patient said the consultation fee was too high.",
    "recommended_action": "Inform patient about affordable slots. "
}

try:
  FollowupInsight(**bad_confidence)
except ValidationError as e:
  print("Test 1 (confidence=1.7) error:\n", e, "\n")

missing_action = {
    "root_cause": "work_or_schedule",
    "confidence": 0.85,
    "summary": "Patient had an urgent office meeting."
}

try:
  FollowupInsight(**missing_action)
except ValidationError as e:
  print("Test 2 (missing recommended_action) error:\n", e, "\n")

# Compare V1 vs V2 prompts on N007 (Hinglish) and N009 (ambiguous)
print("(b) Prompt Comparison: V1 vs V2")

def run_prompt_test(prompt_text):
  global api_call_count
  api_call_count += 1
  response = client.models.generate_content(
      model = MODEL_NAME,
      contents = prompt_text,
      config = types.GenerateContentConfig(
          system_instruction = SYSTEM_TEXT,
          temperature = 0,
          max_output_tokens = 1000,
          response_mime_type = "application/json",
          response_schema = SCHEMA,
          thinking_config = types.ThinkingConfig(thinking_level="minimal")
      )
  )
  return json.loads(response.text)

comparison_rows = []
for target_id in ["N007", "N009"]:
  note_txt = notes_df.loc[notes_df["note_id"] == target_id, "note_text"].iloc[0]

  v1_prompt = f'Classify this note: {note_txt}'
  v1_out = run_prompt_test(v1_prompt)
  time.sleep(13)

  v2_prompt = build_craft_prompt(note_txt)
  v2_out = run_prompt_test(v2_prompt)
  time.sleep(13)

  comparison_rows.append({
      "note_id": target_id,
      "prompt": "V1 (Simple)",
      "root_cause": v1_out.get("root_cause"),
      "in_allowed_list": v1_out.get("root_cause") in ALLOWED_CAUSES,
      "confidence": v1_out.get("confidence")
  })

  comparison_rows.append({
      "note_id": target_id,
      "prompt": "V2 (CRAFT)",
      "root_cause": v2_out.get("root_cause"),
      "in_allowed_list": v2_out.get("root_cause") in ALLOWED_CAUSES,
      "confidence": v2_out.get("confidence")
  })

comp_df = pd.DataFrame(comparison_rows)
display(comp_df)

print(
    "\nDeployment Decision: Deploy V2 (CRAFT prompt). Without explicit allowed labels and "
    "ambiguity rules, V1 invents free-text categories that fail downstream validation, "
    "whereas V2 handles Hinglish (N007) cleanly and safely falls back to 'other' on ambiguous notes (N009)."
)

# Total number of API calls from Task 4.1 - 4.3
print(f"\nTotal Gemini API calls used across Tasks 4.1-4.3: {api_call_count}")

 (a) Pydantic Validation Tests
Test 1 (confidence=1.7) error:
 1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal 

Test 2 (missing recommended_action) error:
 1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'work_or_s...urgent office meeting.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing 

(b) Prompt Comparison: V1 vs V2


,note_id,prompt,root_cause,in_allowed_list,confidence
0,N007,V1 (Simple),Communication failure / Patient forgetfulness,False,0.88
1,N007,V2 (CRAFT),forgot_or_no_reminder,True,0.95
2,N009,V1 (Simple),Unreachable Patient - Phone Switched Off,False,0.95
3,N009,V2 (CRAFT),other,True,0.20



Deployment Decision: Deploy V2 (CRAFT prompt). Without explicit allowed labels and ambiguity rules, V1 invents free-text categories that fail downstream validation, whereas V2 handles Hinglish (N007) cleanly and safely falls back to 'other' on ambiguous notes (N009).

Total Gemini API calls used across Tasks 4.1-4.3: 14


## Task 4.4 — Recommendation and GitHub

In [7]:
# Merge notes_analysed.csv with appointments_clean.csv
appts_clean = pd.read_csv("appointments_clean.csv")
notes_final = pd.read_csv("notes_analysed.csv")

merged_notes = notes_final.merge(appts_clean, on="appointment_id", how="inner")
print(f"Merged table rows: {len(merged_notes)}")

ct = pd.crosstab(merged_notes["root_cause"], merged_notes["reminder_sent"], margins = True)
print("\nCross-tabulation: root_cause vs reminder_sent")
display(ct)

Merged table rows: 10

Cross-tabulation: root_cause vs reminder_sent


reminder_sent,No,Yes,All
root_cause,,,
cost_or_fee,0,1,1
distance_or_transport,1,1,2
forgot_or_no_reminder,2,0,2
illness_or_emergency,0,1,1
other,1,1,2
work_or_schedule,1,1,2
All,5,5,10
